# Diferencijalna jednačina - Masa-opruga-prigušivač sistem

Numeričko rešavanje diferencijalne jednačine drugog reda za sistem sa prinudnim oscilacijama.

Uvoz paketa, konstanti i rešavanje jednačine (definisano u `simulacija.py`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from simulacija import T_odabiranja, f_odabiranja, resi, t

rezultat = resi()

## Opis jednačine:

Diferencijalna jednačina sistema:
\begin{align}
M x'' + \mu x' + c x & = F = A P \sin(2 \pi f t) \\
\end{align}

Transformacija u sistem prvog reda za numeričko rešavanje:
\begin{align}
x'(t) & = v(t) \\
v'(t) & = (F - \mu v(t) - c x) / M \\
\end{align}

## Iscrtavanje odziva

In [ ]:
# Pomeranje i brzina imaju različite jedinice, pa se prikazuju odvojeno.
fig, ose = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)

ax_x, ax_v, ax_pocetak, ax_ustaljeno = ose.flat
ax_x.plot(t, rezultat[:, 0], color='tab:blue')
ax_x.set(xlabel='t [s]', ylabel='x(t) [m]', title='Pomeranje')

ax_v.plot(t, rezultat[:, 1], color='tab:green')
ax_v.set(xlabel='t [s]', ylabel='v(t) [m/s]', title='Brzina')

maska_pocetka = t < 0.1
ax_pocetak.plot(t[maska_pocetka], rezultat[maska_pocetka, 1], color='tab:green')
ax_pocetak.set(xlabel='t [s]', ylabel='v(t) [m/s]', title='Prolazni režim')

maska_kraja = t >= t[-1] - 1.0
ax_ustaljeno.plot(t[maska_kraja], rezultat[maska_kraja, 1], color='tab:green')
ax_ustaljeno.set(xlabel='t [s]', ylabel='v(t) [m/s]', title='Ustaljeni režim')

for osa in ose.flat:
    osa.grid(alpha=0.3)

plt.show()

### Zaključci:
- Primećuju se oscilacije u brzini na početku (prolazni režim)
- Slobodne oscilacije slabe, dok harmonijska pobuda održava oscilacije u ustaljenom režimu

## Amplitudski spektar brzine

Za poslednju sekundu odziva računa se jednostrani amplitudski spektar:

In [ ]:
# Jednostrani amplitudski spektar poslednje sekunde (ustaljeni režim)
brzina_ustaljeno = rezultat[-f_odabiranja:, 1]
broj_odbiraka = len(brzina_ustaljeno)
spektar = np.fft.rfft(brzina_ustaljeno)
freq = np.fft.rfftfreq(broj_odbiraka, d=T_odabiranja)
amplitudski_spektar = 2 * np.abs(spektar) / broj_odbiraka

# DC, a kod parnog broja odbiraka i Nyquist komponenta, nemaju par.
amplitudski_spektar[0] /= 2
if broj_odbiraka % 2 == 0:
    amplitudski_spektar[-1] /= 2

plt.figure(figsize=(15, 7))
plt.semilogy(freq[1:], amplitudski_spektar[1:], linewidth=2)
plt.xlabel('f [Hz]')
plt.ylabel('Amplituda brzine [m/s]')
plt.title('Amplitudski spektar brzine u ustaljenom režimu')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Prikazuju se samo lokalni maksimumi veći od 1% najjače AC komponente.
prag = 0.01 * np.max(amplitudski_spektar[1:])
lokalni_maksimumi = np.flatnonzero(
    (amplitudski_spektar[1:-1] > amplitudski_spektar[:-2])
    & (amplitudski_spektar[1:-1] >= amplitudski_spektar[2:])
) + 1
znacajni_maksimumi = lokalni_maksimumi[
    amplitudski_spektar[lokalni_maksimumi] >= prag
]

for indeks in znacajni_maksimumi:
    print(
        f'Frekvencija = {freq[indeks]:.2f} Hz, '
        f'vršna amplituda brzine = {amplitudski_spektar[indeks]:.6e} m/s'
    )